# Concept-aware pruning with `concept_prune`

Prunes Llama-3.2-1B's MLP width by 40% using concept-aware scores, then compares it with magnitude pruning at the same size.

**Run on one GPU only.** On this server, copies between GPU 0 and GPU 1 are corrupted, so a model split across both GPUs gives NaN. The first cell pins everything to GPU 0. If you change it, restart the kernel, because it must run before `torch` is imported.

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"          # one GPU: cross-GPU copies are broken on this server
os.environ["TOKENIZERS_PARALLELISM"] = "false"    # silences fork warnings during lm_eval

In [2]:
import random
import torch
from concept_prune import load_model, find_decoder_layers
from concept_prune.concepts import concepts_from_templates
from concept_prune.examples_templates import generate_ioi, generate_sva
from concept_prune.attribution import compute_concept_importance, combine_concepts
from concept_prune.pruning import prune_model

/mnt/extra/prune/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load the model

In [3]:
# fp16 matches the research report; the package default is bf16.
model, tok = load_model("meta-llama/Llama-3.2-1B", dtype=torch.float16, device_map={"": 0})
layers     = find_decoder_layers(model)
print(model.dtype, sorted({str(p.device) for p in model.parameters()}))

torch.float16 ['cuda:0']


## 2. Concept probes and attribution

In [4]:
random.seed(0)   # the built-in templates are random; seed them so runs repeat
concepts = concepts_from_templates(tok, {"ioi": generate_ioi, "sva": generate_sva}, n_per_concept=60)

[concepts] ioi: 60 valid pairs (of 180 candidates)
[concepts] sva: 60 valid pairs (of 180 candidates)


In [5]:
coverage = combine_concepts({n: compute_concept_importance(model, p, layers=layers) for n, p in concepts.items()})

assert all(torch.isfinite(v).all() for v in coverage.values()), \
    "coverage has NaN/inf: check the model is on a single GPU (first cell)"
print(f"coverage OK: {len(coverage)} layers, layer 0 range [{coverage[0].min():.3g}, {coverage[0].max():.3g}]")

[attribution] 60/60 pairs used
[attribution] 60/60 pairs used
coverage OK: 16 layers, layer 0 range [0.000292, 0.125]


## 3. Prune: concept-aware and the magnitude baseline

In [6]:
pruned = prune_model(model, 0.4, coverage_by_layer=coverage, beta=2.0)

baseline = prune_model(model, 0.4, coverage_by_layer=None, beta=0.0)
count = lambda m: sum(p.numel() for p in m.parameters())
print(f"dense {count(model)/1e6:.1f}M | baseline {count(baseline)/1e6:.1f}M | concept-aware {count(pruned)/1e6:.1f}M")

dense 1235.8M | baseline 913.8M | concept-aware 913.8M


## 4. Evaluate

Held-out concept probes use names, nouns and countries that the attribution step never saw. The package's built-in templates reuse the same word lists every time, so they can't be used for an honest held-out score.

In [7]:
import random
from concept_prune.concepts import ConceptPair

H_NAMES = ["Charles", "Nancy", "Joseph", "Betty", "Daniel", "Sandra", "Matthew", "Margaret",
           "Anthony", "Ashley", "Kevin", "Kimberly", "Steven", "Emily", "Edward", "Donna"]
H_PLACES = ["bakery", "school", "airport", "theater", "harbor", "clinic", "stadium", "office"]
H_ITEMS = ["photo", "map", "watch", "ticket", "candle", "badge", "ring", "note"]
H_NOUNS = [("boxes", "box"), ("cats", "cat"), ("tables", "table"), ("windows", "window"),
           ("planes", "plane"), ("doctors", "doctor"), ("drivers", "driver"), ("cameras", "camera")]
H_CONTAINERS = ["closet", "kitchen", "yard", "basement", "attic", "hallway", "warehouse", "studio"]
H_CAPITALS = [("Norway", "Oslo"), ("Sweden", "Stockholm"), ("Finland", "Helsinki"), ("Austria", "Vienna"),
              ("Belgium", "Brussels"), ("Ireland", "Dublin"), ("Hungary", "Budapest"), ("Romania", "Bucharest"),
              ("Turkey", "Ankara"), ("Mexico", "Mexico City"), ("Brazil", "Brasilia"), ("Argentina", "Buenos Aires"),
              ("Kenya", "Nairobi"), ("Morocco", "Rabat"), ("Thailand", "Bangkok"), ("Vietnam", "Hanoi"),
              ("Peru", "Lima"), ("Chile", "Santiago"), ("Denmark", "Copenhagen"), ("Ukraine", "Kyiv")]

def holdout_ioi(n):
    out = []
    for _ in range(n):
        a, b = random.sample(H_NAMES, 2); p = random.choice(H_PLACES); it = random.choice(H_ITEMS)
        out.append(ConceptPair(f"{a} and {b} went to the {p}. {a} gave a {it} to",
                               f"{a} and {b} went to the {p}. {b} gave a {it} to", " " + b))
    return out

def holdout_sva(n):
    out = []
    for _ in range(n):
        pl, sg = random.choice(H_NOUNS); c = random.choice(H_CONTAINERS)
        out.append(ConceptPair(f"The {pl} in the {c}", f"The {sg} in the {c}", " are"))
    return out

def holdout_factual(n):
    out = []
    for _ in range(n):
        (c1, cap1), (c2, _) = random.choice(H_CAPITALS), random.choice(H_CAPITALS)
        if c1 != c2:
            out.append(ConceptPair(f"The capital of {c1} is", f"The capital of {c2} is", " " + cap1))
    return out

random.seed(1)
holdout = concepts_from_templates(tok, {"ioi": holdout_ioi, "sva": holdout_sva, "factual": holdout_factual},
                                  n_per_concept=60)

[concepts] ioi: 60 valid pairs (of 180 candidates)
[concepts] sva: 60 valid pairs (of 180 candidates)
[concepts] factual: 60 valid pairs (of 168 candidates)


In [8]:
import numpy as np
import torch
from datasets import load_dataset
from torch.utils.data import DataLoader

def wiki_loader(n=100, max_length=1024, batch_size=4):
    ds = load_dataset("wikitext", "wikitext-2-raw-v1", split=f"test[:{n}]")
    ds = ds.map(lambda ex: tok(ex["text"], truncation=True, padding="max_length", max_length=max_length),
                batched=True, remove_columns=ds.column_names)
    ds.set_format(type="torch", columns=["input_ids", "attention_mask"])
    return DataLoader(ds, batch_size=batch_size)

@torch.no_grad()
def perplexity(m, loader):
    dev = next(m.parameters()).device
    total, tokens = 0.0, 0
    for b in loader:
        ids, am = b["input_ids"].to(dev), b["attention_mask"].to(dev)
        labels = ids.clone(); labels[am == 0] = -100          # padding is not scored
        n = am[..., 1:].sum().item()
        total += m(input_ids=ids, attention_mask=am, labels=labels).loss.item() * n
        tokens += n
    return float(np.exp(total / tokens))

wiki = wiki_loader()

Map: 100%|██████████| 100/100 [00:00<00:00, 1897.90 examples/s]


In [ ]:
import pandas as pd
from concept_prune.evaluation import evaluate_concept_accuracy, evaluate_lm_eval

TASKS = ["lambada_openai", "arc_easy", "piqa"]   # add hellaswag, winogrande if you have time
LIMIT = None #200                                      # examples per task; None = full test sets (slow)

def evaluate(m, name):
    row = {"method": name, "params (M)": round(count(m) / 1e6, 1), "wiki ppl": perplexity(m, wiki)}
    row.update({f"{c} acc": v for c, v in evaluate_concept_accuracy(m, holdout).items()})
    for task, r in evaluate_lm_eval(m, tok, TASKS, limit=LIMIT).items():
        if "perplexity,none" in r:
            row[f"{task} ppl"] = r["perplexity,none"]
        row[f"{task} acc"] = r.get("acc,none")
    return row

results = pd.DataFrame([evaluate(model, "dense"),
                        evaluate(baseline, "magnitude"),
                        evaluate(pruned, "concept-aware (β=2)")]).set_index("method")
results.round(3)

`pretrained` model kwarg is not of type `str`. Many other model arguments may be ignored. Please do not launch via accelerate or use `parallelize=True` if passing an existing model this way.
Passed an already-initialized model through `pretrained`, assuming single-process call to evaluate() or custom distributed integration
Overwriting default num_fewshot of piqa from None to 0
Overwriting default num_fewshot of arc_easy from None to 0
Overwriting default num_fewshot of lambada_openai from None to 0
Running loglikelihood requests: 100%|██████████| 1399/1399 [00:05<00:00, 244.83it/s]


bootstrapping for stddev: perplexity


100%|██████████| 100/100 [00:00<00:00, 842.39it/s]
`pretrained` model kwarg is not of type `str`. Many other model arguments may be ignored. Please do not launch via accelerate or use `parallelize=True` if passing an existing model this way.
Passed an already-initialized model through `pretrained`, assuming single-process call to evaluate() or custom distributed integration
Overwriting default num_fewshot of piqa from None to 0
Overwriting default num_fewshot of arc_easy from None to 0
Overwriting default num_fewshot of lambada_openai from None to 0
Running loglikelihood requests: 100%|██████████| 1399/1399 [00:05<00:00, 236.57it/s]


bootstrapping for stddev: perplexity


100%|██████████| 100/100 [00:00<00:00, 936.62it/s]
`pretrained` model kwarg is not of type `str`. Many other model arguments may be ignored. Please do not launch via accelerate or use `parallelize=True` if passing an existing model this way.
Passed an already-initialized model through `pretrained`, assuming single-process call to evaluate() or custom distributed integration
Overwriting default num_fewshot of piqa from None to 0
Overwriting default num_fewshot of arc_easy from None to 0
Overwriting default num_fewshot of lambada_openai from None to 0
Running loglikelihood requests: 100%|██████████| 1399/1399 [00:05<00:00, 259.17it/s]


bootstrapping for stddev: perplexity


100%|██████████| 100/100 [00:00<00:00, 940.30it/s]


,params (M),wiki ppl,ioi acc,sva acc,factual acc,arc_easy acc,lambada_openai ppl,lambada_openai acc,piqa acc
method,,,,,,,,,
dense,1235.8,24.762,0.783,0.617,0.750,0.65,5.626,0.635,0.730
magnitude,913.8,109.503,0.083,0.533,0.000,0.40,88.490,0.300,0.625
concept-aware (β=2),913.8,90.129,1.000,0.817,0.567,0.42,24.956,0.435,0.635
